# Check length for all articles

In [ ]:
import os
import glob
import re
import pandas as pd

folder_path = "ESG News Data ProQuest"
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))


csv_files = sorted(
    csv_files,
    key=lambda x: int(re.search(r"EsgNews(\d+)\.csv", os.path.basename(x)).group(1))
)

for file in csv_files:
    df = pd.read_csv(file)
    
    print(f"\n {os.path.basename(file)}")
    
    for idx, text in df["text"].fillna("").astype(str).items():
        Length = len(text)
        print(f"Article {idx}: {Length}")


 EsgNews1.csv
Article 0: 5731
Article 1: 16616
Article 2: 9619
Article 3: 4251
Article 4: 5115
Article 5: 9908
Article 6: 6907
Article 7: 8690
Article 8: 6907
Article 9: 6907
Article 10: 6899
Article 11: 4511
Article 12: 7536
Article 13: 12613
Article 14: 8127
Article 15: 13287
Article 16: 12226
Article 17: 12227
Article 18: 6372
Article 19: 4170

 EsgNews2.csv
Article 0: 6503
Article 1: 7074
Article 2: 3600
Article 3: 4752
Article 4: 8892
Article 5: 3750
Article 6: 11813
Article 7: 6293
Article 8: 6900
Article 9: 62986
Article 10: 7485
Article 11: 7196
Article 12: 8830
Article 13: 16264
Article 14: 11099
Article 15: 8962
Article 16: 14492
Article 17: 7096
Article 18: 7285
Article 19: 18160

 EsgNews3.csv
Article 0: 9227
Article 1: 12299
Article 2: 4390
Article 3: 9029
Article 4: 6818
Article 5: 16758
Article 6: 12834
Article 7: 12116
Article 8: 4013
Article 9: 7919
Article 10: 6237
Article 11: 8617
Article 12: 12361
Article 13: 10144
Article 14: 6026
Article 15: 7727
Article 16: 1069

# Filtering out long articles (length > 13,000)

In [ ]:


input_folder = "ESG News Data ProQuest"
output_folder = "ESG_Dataset_13000"

os.makedirs(output_folder, exist_ok=True)

csv_files = glob.glob(os.path.join(input_folder, "*.csv"))

csv_files = sorted(
    csv_files,
    key=lambda x: int(re.search(r"EsgNews(\d+)\.csv", os.path.basename(x)).group(1))
)

for file in csv_files:
    df = pd.read_csv(file)

    # Replace missing values in the text column
    df["text"] = df["text"].fillna("").astype(str)

    # Count the number of characters in each article
    df["char_count"] = df["text"].apply(len)

    # Keep only rows where text length is 13000 characters or fewer
    df_filtered = df[df["char_count"] <= 13000].copy()
    df_filtered = df_filtered.drop(columns=["char_count"])

    output_path = os.path.join(output_folder, os.path.basename(file))
    df_filtered.to_csv(output_path, index=False)

    print(f"{os.path.basename(file)}: original {len(df)} rows, kept {len(df_filtered)} rows, removed {len(df) - len(df_filtered)} rows")

print("All files have been processed.")

EsgNews1.csv: original 20 rows, kept 18 rows, removed 2 rows
EsgNews2.csv: original 20 rows, kept 16 rows, removed 4 rows
EsgNews3.csv: original 20 rows, kept 18 rows, removed 2 rows
EsgNews4.csv: original 20 rows, kept 17 rows, removed 3 rows
EsgNews5.csv: original 20 rows, kept 17 rows, removed 3 rows
EsgNews6.csv: original 20 rows, kept 10 rows, removed 10 rows
EsgNews7.csv: original 20 rows, kept 14 rows, removed 6 rows
EsgNews8.csv: original 20 rows, kept 17 rows, removed 3 rows
EsgNews9.csv: original 20 rows, kept 13 rows, removed 7 rows
EsgNews10.csv: original 20 rows, kept 20 rows, removed 0 rows
EsgNews11.csv: original 20 rows, kept 19 rows, removed 1 rows
EsgNews12.csv: original 19 rows, kept 18 rows, removed 1 rows
EsgNews13.csv: original 20 rows, kept 17 rows, removed 3 rows
EsgNews14.csv: original 20 rows, kept 16 rows, removed 4 rows
EsgNews15.csv: original 19 rows, kept 16 rows, removed 3 rows
EsgNews16.csv: original 20 rows, kept 17 rows, removed 3 rows
EsgNews17.csv: o

# Remove duplicated articles

In [ ]:

input_folder = "ESG_Dataset_13000"
output_file = "ESG_Dataset_13000_3.25/all_articles_title_dedup.csv"

csv_files = glob.glob(os.path.join(input_folder, "*.csv"))

csv_files = sorted(
    [f for f in csv_files if re.search(r"EsgNews(\d+)\.csv$", os.path.basename(f))],
    key=lambda x: int(re.search(r"EsgNews(\d+)\.csv$", os.path.basename(x)).group(1))
)

all_dfs = []

for file in csv_files:
    df = pd.read_csv(file)

    if "document_title" not in df.columns or "publication_date" not in df.columns:
        print(f"{os.path.basename(file)} skipped: missing 'document_title' or 'publication_date'.")
        continue

    df = df.copy()
    df["source_file"] = os.path.basename(file)
    all_dfs.append(df)

all_articles = pd.concat(all_dfs, ignore_index=True)

print(f"Total rows before title dedup: {len(all_articles)}")

# normalize title slightly for dedup
all_articles["document_title"] = all_articles["document_title"].fillna("").astype(str)
all_articles["title_norm"] = all_articles["document_title"].str.strip().str.lower()

# convert publication_date to datetime
all_articles["publication_date_dt"] = pd.to_datetime(
    all_articles["publication_date"],
    errors="coerce"
)

# sort by earliest publication date first
all_articles = all_articles.sort_values(
    by=["publication_date_dt", "source_file"],
    ascending=[True, True]
).copy()

# keep the earliest article for each title
dedup_articles = all_articles.drop_duplicates(
    subset=["title_norm"],
    keep="first"
).copy()

print(f"Total rows after title dedup: {len(dedup_articles)}")
print(f"Removed duplicate titles: {len(all_articles) - len(dedup_articles)}")

# drop helper columns
dedup_articles = dedup_articles.drop(columns=["title_norm", "publication_date_dt"])

# save combined deduplicated file
dedup_articles.to_csv(output_file, index=False)

print(f"Saved to: {output_file}")


Total rows before title dedup: 2509
Total rows after title dedup: 2194
Removed duplicate titles: 315
Saved to: /Users/helen/BASc/dissertation/coding/ESG_Dataset_13000_3.25/all_articles_title_dedup.csv
